In [1]:
import pandas as pd
import re


In [ ]:
########################################################################################################################################################
# Profiling df function - returns one row per column: how full it is, how many distinct values, its dtype, and whether all non-null values are identical.
########################################################################################################################################################

def profile(frame: pd.DataFrame) -> pd.DataFrame:

    out = pd.DataFrame({
        "non_null": frame.notna().sum(),
        "nulls": frame.isna().sum(),
        "distinct": frame.nunique(dropna=True),
        "dtype": frame.dtypes.astype(str),
    })

    out["pct_null"] = (out["nulls"] / len(frame) * 100).round(1)

    # True if all non-null values in the column are the same
    out["all_values_same"] = out["distinct"] <= 1

    return out.sort_values("non_null", ascending=False)

In [ ]:
########################################################################################################################################################
# Running list of Test Accounts - deletes the patient id's that are known test accounts.
########################################################################################################################################################

def remove_rows_by_values(df, column_name):
    values_to_remove = [
        1, 2, 3, 4, 5, 6, 7, 8,
        22255, 22256, 22258, 22259, 22260, 22261, 22264, 22265,
        22266, 22267, 22268, 22269, 22282, 22315, 22316, 22381,
        22414, 22447, 22448, 22449, 22450, 22451, 22480, 22546,
        # adding ones I found:
        22251, 22254, 22257, 22262, 22263, 22270, 22271, 
        22348, 22415, 22547, 40927, 75412, 111646, 111647, 
        111648, 133921, 133922
    ]

    return df[~df[column_name].isin(values_to_remove)].copy()

In [ ]:
########################################################################################################################################################
# Running list of Physicians - links physician id's with name and id.
########################################################################################################################################################

DOCTOR_LOOKUP = {
    2: {
        "doctor_id": 2,
        "doctor_key": 2,
        "name": "Dr. Alexander Quaas",
        "doctor_group": "RPSD",
    },
    6: {
        "doctor_id": 6,
        "doctor_key": 6,
        "name": "Dr. Sanjay Agarwal",
        "doctor_group": "UCSD",
    },
    5: {
        "doctor_id": 5,
        "doctor_key": 5,
        "name": "Dr. Antoni Duleba",
        "doctor_group": "UCSD",
    },
    43: {
        "doctor_id": 43,
        "doctor_key": 43,
        "name": "Dr. Valerie Flores",
        "doctor_group": "RPSD",
    },
    1: {
        "doctor_id": 1,
        "doctor_key": 1,
        "name": "Dr. Vicente Garzo Toro",
        "doctor_group": "RPSD",
    },
    7: {
        "doctor_id": 7,
        "doctor_key": 7,
        "name": "Dr. Tracy Harrison",
        "doctor_group": "Kaiser",
    },
    7: {
        "doctor_id": 7,
        "doctor_key": 7,
        "name": "Dr. Tracy Harrison",
        "doctor_group": "Kaiser",
    },
    8: {
        "doctor_id": 8,
        "doctor_key": 8,
        "name": "Dr. Li-Shei Lin",
        "doctor_group": "RPSD",
    },
    3: {
        "doctor_id": 3,
        "doctor_key": 3,
        "name": "Dr. Jamie Stanhiser",
        "doctor_group": "RPSD",
    },
    4: {
        "doctor_id": 4,
        "doctor_key": 4,
        "name": "Dr. Hui-Chun Su",
        "doctor_group": "UCSD",
    },
    10: {
        "doctor_id": 10,
        "doctor_key": 10,
        "name": "Dr. Helen Swenson", #Dr. Anglin
        "doctor_group": "Kaiser",
    },
    76: {
        "doctor_id": 76,
        "doctor_key": 76,
        "name": "Dr. Rachel Whynott",
        "doctor_group": "UCSD",
    },
    9: {
        "doctor_id": 9,
        "doctor_key": 9,
        "name": "Dr. Richard Yoo",
        "doctor_group": "Kaiser",
    },
}


def add_doctor_columns(df, doctor_id_column):
    """
    Add standardized doctor information based on a doctor ID column.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    doctor_id_column : str
        Name of the column containing the doctor ID,
        e.g. 'physician' or 'treated_by'.

    Returns
    -------
    pd.DataFrame
        DataFrame with doctor_id, doctor_key, name, and doctor_group.
    """

    df = df.copy()

    # Make sure the doctor ID is numeric
    df["doctor_id"] = pd.to_numeric(
        df[doctor_id_column],
        errors="coerce"
    )

    # Add information from lookup table
    df["doctor_key"] = df["doctor_id"].map(
        lambda x: DOCTOR_LOOKUP.get(x, {}).get("doctor_key")
    )

    df["name"] = df["doctor_id"].map(
        lambda x: DOCTOR_LOOKUP.get(x, {}).get("name")
    )

    df["doctor_group"] = df["doctor_id"].map(
        lambda x: DOCTOR_LOOKUP.get(x, {}).get("doctor_group")
    )

    return df

In [ ]:
########################################################################################################################################################
# Cleaning arbitrary columns function - returns cleaned df without columns that are all empty/all have the same non-null value
########################################################################################################################################################

#def drop_empty_and_constant_columns(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
#    """
#    Parameters
#    ----------
#    df : pd.DataFrame
#        DataFrame to clean.
#    verbose : bool, default=True
#        Whether to print a summary of dropped columns.
#    """
#    prof = profile(df)
#
#    cols_to_drop = prof.index[
#        (prof["non_null"] == 0) | (prof["all_values_same"])
#    ].tolist()
#
#    if verbose:
#        print(f"Dropping {len(cols_to_drop)} empty or constant columns:\n")
#        for c in cols_to_drop:
#            print(f"  - {c}")
#
#    cleaned_df = df.drop(columns=cols_to_drop)
#
#    if verbose:
#        print(f"\nRemaining: {cleaned_df.shape[1]} columns")
#
#    return cleaned_df


In [ ]:
########################################################################################################################################################
# Cleaning arbitrary columns function - returns cleaned df without columns that are all empty/all have the same non-null value
########################################################################################################################################################

def drop_empty_and_constant_columns(
    df: pd.DataFrame,
    prof: pd.DataFrame | None = None,
    verbose: bool = True,
) -> pd.DataFrame:
    if prof is None:
        prof = profile(df)

    cols_to_drop = prof.index[
        (prof["non_null"] == 0) | (prof["all_values_same"])
    ].tolist()

    if verbose:
        print(f"Dropping {len(cols_to_drop)} columns")

    return df.drop(columns=cols_to_drop)


In [ ]:
########################################################################################################################################################
# add_oocyte_features
# ---------------------------------------------------------------------------
# Extracts binary morphology and injection-behavior features from free-text
# embryology fields (oolemma, cyto, cumulus, pv_space). Each feature = 1 if
# the corresponding pattern appears in ANY of the listed columns for that row.
#
# Features extracted:
#
#   Cytoplasmic morphology:
#     granular              - granular cytoplasm
#     vacuoles              - vacuoles present ("vac", "vacuol")
#     fragments             - cytoplasmic fragments
#     inclusions            - cytoplasmic inclusions
#     cldga                 - centrally located granular area
#     degenerating          - degeneration / atresia / lysing
#     ser                   - smooth endoplasmic reticulum aggregate
#     refractile_body       - refractile body ("RB")
#     dark                  - dark cytoplasm (excludes "dark inclusion" /
#                             "dark fragment" to avoid double-counting)
#
#   Oolemma / injection behavior:
#     retracted             - oolemma retracts from needle (too elastic)
#     no_resistance         - oolemma breaks too easily (fragile/flaccid)
#     sticky_or_difficult   - sticky ooplasm and/or difficult to inject
#                             (combined; these nearly always co-occur)
#
#   Perivitelline space / polar body:
#     large_pvs             - enlarged perivitelline space ("Lg PVS")
#     dark_pvs              - dark/debris-filled PVS ("DPVS", incl. "DVPS" typo)
#     fragmented_pb         - fragmented polar body ("f PB", "fPB", etc.)
#     multiple_pb           - more than one polar body present
#     abnormal_pb_size      - unusually large or small polar body
#     ghost_pb              - faint/degenerated polar body remnant
#     detached_pb           - polar body separated from oolemma
#     pvs_debris            - debris or fragments in the PVS
#     possible_parth        - suspected parthenogenetic activation
#     pb_je                 - polar body "just extruded" (observation note,
#                             not necessarily pathological)
#
# Notes:
#   - Text is lowercased and periods stripped before matching.
#   - Some features may co-fire on the same row (e.g. "f PB JE" hits both fragmented_pb and pb_je) — this is intentional.
#   - pb_je is a timing/observation note, not a morphology defect; consider excluding it from downstream risk models.
######################################################################################################################################################################################################################################

def add_oocyte_features(df, cols=("oolemma", "cyto", "cumulus", "pv_space")):
    """
    Extract binary morphology / injection features from free-text fields.
    Searches across all listed columns (default: oolemma, cyto, cumulus, pv_space).
    Each feature = 1 if the pattern appears in ANY of the columns.
    """
    # Combine text from all specified columns into one searchable field
    combined = pd.Series("", index=df.index, dtype=str)
    for col in cols:
        if col in df.columns:
            piece = (
                df[col]
                .fillna("")
                .astype(str)
                .str.lower()
                .str.replace(".", "", regex=False)
            )
            combined = combined.str.cat(piece, sep=" ")
    text = combined

    # --- cytoplasmic morphology ---
    df["granular"]     = text.str.contains(
        r"granular"
        ).astype(int)
    df["vacuoles"]     = text.str.contains(
        r"vac|vacuol"
        ).astype(int)
    df["fragments"]    = text.str.contains(
        r"fragment"
        ).astype(int)
    df["inclusions"]   = text.str.contains(
        r"inclusion"
        ).astype(int)
    df["cldga"]        = text.str.contains(
        r"cldga"
        ).astype(int)
    df["degenerating"] = text.str.contains(
        r"deg|degenerat|atretic|lysing"
        ).astype(int)
    df["ser"]          = text.str.contains(
        r"\bser\b"
        ).astype(int)
    df["refractile_body"] = text.str.contains(
        r"\brb\b"
        ).astype(int)
    df["dark"] = (
        text.str.contains(
            r"dark|\bdk\b")     # dark, but exclude dark inclusions / dark fragments; also catch "dk"
        & ~text.str.contains(r"dark inclusion|dark fragment|dk inclusion|dk fragment")
    ).astype(int)
    df["retracted"]     = text.str.contains(
        r"retract"
        ).astype(int)
    df["no_resistance"] = text.str.contains(
        r"no resistance"
        ).astype(int)
    df["sticky_or_difficult"] = text.str.contains(
        r"sticky|diff|hard to inject"     # sticky ooplasm and/or difficult to inject -> combined flag
    ).astype(int)
    df["large_pvs"] = text.str.contains(
        r"lg pvs|large pv|lgpvs"     # enlarged perivitelline space
        ).astype(int)
    df["dark_pvs"] = text.str.contains(
        r"dpvs|dvps|dark pvs"     # debris/darkness in PVS; DVPS is a common typo of DPVS
        ).astype(int)
    df["fragmented_pb"] = text.str.contains(
        r"\bf ?pb|\bfpb"     # fragmented polar body: "f PB", "fPB", "f PBJE", etc.
        ).astype(int)
    df["multiple_pb"] = text.str.contains(
        r"multi ?pb|multipb|mult pb|mulit pb|multpb|\bmult ?lg pb|2 ?pb|3 ?pb|4 ?pb|5 ?pb|numerous pb|number pb|multiple pb"
        ).astype(int)
    df["abnormal_pb_size"] = text.str.contains(
        r"lg pb|large pb|sm pb|small pb|giant pb|very lg pb|lrg pb|lgpb"     # size deviations from normal
        ).astype(int)
    df["ghost_pb"] = text.str.contains(
        r"ghost pb|ghost f pb|ghost fpb"     # faint/degenerated polar body remnant
        ).astype(int)
    df["detached_pb"] = text.str.contains(
        r"detached pb|dislodged pb|floating pb|floating fpb|floating f pb"     # PB separated from oolemma
        ).astype(int)
    df["pvs_debris"] = text.str.contains(
        r"debris|frags"     # debris or fragments in PVS
        ).astype(int)
    df["possible_parth"] = text.str.contains(
        r"parth"     # parthenogenesis suspected
        ).astype(int)
    df["pb_je"] = text.str.contains(
        r"pb ?je|pbje|f ?pbje"     # polar body just extruded
        ).astype(int)

    return df

In [ ]:
########################################################################################################################################################
# total_eggs_ICSId = max ord per cycleid
#
#total_day5_G = sum of all day 5 cases (rows) per cycleid where grade ends in "G"
#otal_day5_F = sum of all day 5 cases (rows) per cycleid where grade ends in "F"
#total_day5_F-P = sum of all day 5 cases (rows) per cycleid where grade ends in "F-P"
#total_day5_P = sum of all day 5 cases (rows) per cycleid where grade ends in "P"
#total_day6_G = sum of all day 6 cases (rows) per cycleid where grade ends in "G"
#total_day6_F = sum of all day 6 cases (rows) per cycleid where grade ends in "F"
#total_day6_F-P = sum of all day 6 cases (rows) per cycleid where grade ends in "F-P"
#total_day6_P = sum of all day 6 cases (rows) per cycleid where grade ends in "P"
#total_day7_G = sum of all day 7 cases (rows) per cycleid where grade ends in "G"
#total_day7_F = sum of all day 7 cases (rows) per cycleid where grade ends in "F"
#total_day7_F-P = sum of all day 7 cases (rows) per cycleid where grade ends in "F-P"
#total_day7_P = sum of all day 7 cases (rows) per cycleid where grade ends in "P"
#
#total_day_5_euploid = sum of all cases (rows) per cycleid where max_day is 5 and pgd_result contains string "euploid" (upper- or lowercase)
#total_day_5_aneuploid = sum of all cases (rows) per cycleid where max_day is 5 and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day_6_euploid = sum of all cases (rows) per cycleid where max_day is 6 and pgd_result contains string "euploid" (upper- or lowercase)
#total_day_6_aneuploid = sum of all cases (rows) per cycleid where max_day is 6 and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day_7_euploid = sum of all cases (rows) per cycleid where max_day is 7 and pgd_result contains string "euploid" (upper- or lowercase)
#total_day_7_aneuploid = sum of all cases (rows) per cycleid where max_day is 7 and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_untested = sum of all cases (rows) per cycleid where grade is AAG, ABG, BAG, BBF, ACF, CAF, BCF-P, CBF-P, BBF-P, CCP, BCP, CBP, CCP, XXP, or XXF and pgd_result is NA
#
#total_day5_euploid_AAG = sum of cases (rows) per cycleid where max_day is 5 and grade is AAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_AAG = sum of cases (rows) per cycleid where max_day is 5 and grade is AAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_ABG = sum of cases (rows) per cycleid where max_day is 5 and grade is ABG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_ABG = sum of cases (rows) per cycleid where max_day is 5 and grade is ABG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_BAG = sum of cases (rows) per cycleid where max_day is 5 and grade is BAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_BAG = sum of cases (rows) per cycleid where max_day is 5 and grade is BAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_BBF = sum of cases (rows) per cycleid where max_day is 5 and grade is BBF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_BBF = sum of cases (rows) per cycleid where max_day is 5 and grade is BBF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_ACF = sum of cases (rows) per cycleid where max_day is 5 and grade is ACF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_ACF = sum of cases (rows) per cycleid where max_day is 5 and grade is ACF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_CAF = sum of cases (rows) per cycleid where max_day is 5 and grade is CAF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_CAF = sum of cases (rows) per cycleid where max_day is 5 and grade is CAF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_BCF-P = sum of cases (rows) per cycleid where max_day is 5 and grade is BCF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_BCF-P = sum of cases (rows) per cycleid where max_day is 5 and grade is BCF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_CBF-P = sum of cases (rows) per cycleid where max_day is 5 and grade is CBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_CBF-P = sum of cases (rows) per cycleid where max_day is 5 and grade is CBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_CCP = sum of cases (rows) per cycleid where max_day is 5 and grade is CCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_CCP = sum of cases (rows) per cycleid where max_day is 5 and grade is CCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_BCP = sum of cases (rows) per cycleid where max_day is 5 and grade is BCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_BCP = sum of cases (rows) per cycleid where max_day is 5 and grade is BCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_CBP = sum of cases (rows) per cycleid where max_day is 5 and grade is CBP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_CBP = sum of cases (rows) per cycleid where max_day is 5 and grade is CBP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_BBF-P = sum of cases (rows) per cycleid where max_day is 5 and grade is BBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_BBF-P = sum of cases (rows) per cycleid where max_day is 5 and grade is BBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_XXF = sum of cases (rows) per cycleid where max_day is 5 and grade is XXF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_XXF = sum of cases (rows) per cycleid where max_day is 5 and grade is XXF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day5_euploid_XXP = sum of cases (rows) per cycleid where max_day is 5 and grade is XXP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day5_aneuploid_XXP = sum of cases (rows) per cycleid where max_day is 5 and grade is XXP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#
#total_day6_euploid_AAG = sum of cases (rows) per cycleid where max_day is 6 and grade is AAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_AAG = sum of cases (rows) per cycleid where max_day is 6 and grade is AAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_ABG = sum of cases (rows) per cycleid where max_day is 6 and grade is ABG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_ABG = sum of cases (rows) per cycleid where max_day is 6 and grade is ABG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_BAG = sum of cases (rows) per cycleid where max_day is 6 and grade is BAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_BAG = sum of cases (rows) per cycleid where max_day is 6 and grade is BAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_BBF = sum of cases (rows) per cycleid where max_day is 6 and grade is BBF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_BBF = sum of cases (rows) per cycleid where max_day is 6 and grade is BBF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_ACF = sum of cases (rows) per cycleid where max_day is 6 and grade is ACF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_ACF = sum of cases (rows) per cycleid where max_day is 6 and grade is ACF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_CAF = sum of cases (rows) per cycleid where max_day is 6 and grade is CAF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_CAF = sum of cases (rows) per cycleid where max_day is 6 and grade is CAF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_BCF-P = sum of cases (rows) per cycleid where max_day is 6 and grade is BCF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_BCF-P = sum of cases (rows) per cycleid where max_day is 6 and grade is BCF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_CBF-P = sum of cases (rows) per cycleid where max_day is 6 and grade is CBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_CBF-P = sum of cases (rows) per cycleid where max_day is 6 and grade is CBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_CCP = sum of cases (rows) per cycleid where max_day is 6 and grade is CCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_CCP = sum of cases (rows) per cycleid where max_day is 6 and grade is CCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_BCP = sum of cases (rows) per cycleid where max_day is 6 and grade is BCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_BCP = sum of cases (rows) per cycleid where max_day is 6 and grade is BCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_CBP = sum of cases (rows) per cycleid where max_day is 6 and grade is CBP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_CBP = sum of cases (rows) per cycleid where max_day is 6 and grade is CBP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_BBF-P = sum of cases (rows) per cycleid where max_day is 6 and grade is BBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_BBF-P = sum of cases (rows) per cycleid where max_day is 6 and grade is BBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_XXF = sum of cases (rows) per cycleid where max_day is 6 and grade is XXF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_XXF = sum of cases (rows) per cycleid where max_day is 6 and grade is XXF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day6_euploid_XXP = sum of cases (rows) per cycleid where max_day is 6 and grade is XXP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day6_aneuploid_XXP = sum of cases (rows) per cycleid where max_day is 6 and grade is XXP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#
#total_day7_euploid_AAG = sum of cases (rows) per cycleid where max_day is 7 and grade is AAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_AAG = sum of cases (rows) per cycleid where max_day is 7 and grade is AAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_ABG = sum of cases (rows) per cycleid where max_day is 7 and grade is ABG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_ABG = sum of cases (rows) per cycleid where max_day is 7 and grade is ABG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_BAG = sum of cases (rows) per cycleid where max_day is 7 and grade is BAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_BAG = sum of cases (rows) per cycleid where max_day is 7 and grade is BAG (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_BBF = sum of cases (rows) per cycleid where max_day is 7 and grade is BBF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_BBF = sum of cases (rows) per cycleid where max_day is 7 and grade is BBF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_ACF = sum of cases (rows) per cycleid where max_day is 7 and grade is ACF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_ACF = sum of cases (rows) per cycleid where max_day is 7 and grade is ACF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_CAF = sum of cases (rows) per cycleid where max_day is 7 and grade is CAF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_CAF = sum of cases (rows) per cycleid where max_day is 7 and grade is CAF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_BCF-P = sum of cases (rows) per cycleid where max_day is 7 and grade is BCF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_BCF-P = sum of cases (rows) per cycleid where max_day is 7 and grade is BCF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_CBF-P = sum of cases (rows) per cycleid where max_day is 7 and grade is CBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_CBF-P = sum of cases (rows) per cycleid where max_day is 7 and grade is CBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_CCP = sum of cases (rows) per cycleid where max_day is 7 and grade is CCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_CCP = sum of cases (rows) per cycleid where max_day is 7 and grade is CCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_BCP = sum of cases (rows) per cycleid where max_day is 7 and grade is BCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_BCP = sum of cases (rows) per cycleid where max_day is 7 and grade is BCP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_CBP = sum of cases (rows) per cycleid where max_day is 7 and grade is CBP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_CBP = sum of cases (rows) per cycleid where max_day is 7 and grade is CBP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_BBF-P = sum of cases (rows) per cycleid where max_day is 7 and grade is BBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_BBF-P = sum of cases (rows) per cycleid where max_day is 7 and grade is BBF-P (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_XXF = sum of cases (rows) per cycleid where max_day is 7 and grade is XXF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_XXF = sum of cases (rows) per cycleid where max_day is 7 and grade is XXF (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#total_day7_euploid_XXP = sum of cases (rows) per cycleid where max_day is 7 and grade is XXP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "euploid" (upper- or lowercase)
#total_day7_aneuploid_XXP = sum of cases (rows) per cycleid where max_day is 7 and grade is XXP (upper- or lowercase and spacing or no spacing) and pgd_result contains string "aneuploid" (upper- or lowercase)
#
#total_complex_aneuploid = sum of cases (rows) per cycleid where pgd_result contains strings "complex" and "aneuploid" but do not necessarily have to be next to each other (upper- or lowercase)
#total_low_mosaic = sum of cases (rows) per cycleid where pgd_result contain strings "low" and "mosaic" but do not necessarily have to be next to each other (upper- or lowercase)
#total_high_mosaic = sum of cases (rows) per cycleid where pgd_result contains string "high" and mosaic" but do not necessarily have to be next to each other (upper- or lowercase)
#total_chaotic = sum of cases (rows) per cycleid where pgd_result contains string "chaotic"
#total euploid = sum of cases (rows) per cycleid where pgd_result contains "euploid"
#total_aneuploid = sum of cases (rows) per cycleid where pgd_result contains "aneuploid" (except complex aneuploid cases (rows) per cycleid)
#total_blastocysts = sum of all cases (rows) per cycleid where there is a grade of total_untested = sum of all cases (rows) per cycleid where grade is AAG, ABG, BAG, BBF, ACF, CAF, BCF-P, CBF-P, BBF-P, CCP, BCP, CBP, CCP, XXP, or XXF
######################################################################################################################################################################################################################################

def pivot_merged_df_to_wide(merged_df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot merged_df from long to wide: one row per (patid, cycleid),
    with all the count columns per the spec. Missing counts are 0.

    "normal"   is treated as a synonym for "euploid".
    "abnormal" is treated as a synonym for "aneuploid".
    """
    df = merged_df.copy()

    # ---- Normalizations ----
    grade_up = (
        df['grade'].astype('string')
                   .str.upper()
                   .str.replace(r'\s+', '', regex=True)
                   .fillna('')
    )
    pgd_lc = df['pgd_result'].astype('string').str.lower().fillna('')

    # ---- Grade-suffix masks (mutually exclusive) ----
    ends_FP = grade_up.str.endswith('F-P')
    ends_G  = grade_up.str.endswith('G')
    ends_F  = grade_up.str.endswith('F') & ~ends_FP
    ends_P  = grade_up.str.endswith('P') & ~ends_FP

    # ---- PGD-result masks ----
    # "aneuploid" contains "euploid", and "abnormal" contains "normal",
    # so the base terms exclude their negated counterparts.
    raw_aneuploid = pgd_lc.str.contains('aneuploid', regex=False)
    raw_euploid   = pgd_lc.str.contains('euploid',   regex=False) & ~raw_aneuploid
    raw_abnormal  = pgd_lc.str.contains('abnormal',  regex=False)
    raw_normal    = pgd_lc.str.contains('normal',    regex=False) & ~raw_abnormal

    # Merged synonyms: euploid|normal  and  aneuploid|abnormal
    has_euploid   = raw_euploid   | raw_normal
    has_aneuploid = raw_aneuploid | raw_abnormal

    # "complex" + (aneuploid OR abnormal)
    has_complex_aneu = has_aneuploid & pgd_lc.str.contains('complex', regex=False)

    has_low_mosaic  = pgd_lc.str.contains('low',  regex=False) & pgd_lc.str.contains('mosaic', regex=False)
    has_high_mosaic = pgd_lc.str.contains('high', regex=False) & pgd_lc.str.contains('mosaic', regex=False)
    has_chaotic     = pgd_lc.str.contains('chaotic', regex=False)

    # ---- Blastocyst grades ----
    BLAST_GRADES = ['AAG', 'ABG', 'BAG', 'BBF', 'ACF', 'CAF',
                    'BCF-P', 'CBF-P', 'BBF-P',
                    'CCP', 'BCP', 'CBP', 'XXP', 'XXF']
    is_blast    = grade_up.isin(BLAST_GRADES)
    is_untested = is_blast & df['pgd_result'].isna()

    # ---- Build per-row 0/1 indicators ----
    ind = pd.DataFrame({'patid': df['patid'], 'cycleid': df['cycleid']})
    ind['__ord__'] = df['ord']

    # Day-suffix totals (uses `day`)
    for d in (5, 6, 7):
        day_mask = df['day'] == d
        ind[f'total_day{d}_G']   = (day_mask & ends_G ).astype(int)
        ind[f'total_day{d}_F']   = (day_mask & ends_F ).astype(int)
        ind[f'total_day{d}_F-P'] = (day_mask & ends_FP).astype(int)
        ind[f'total_day{d}_P']   = (day_mask & ends_P ).astype(int)

    # Euploid / aneuploid per max_day
    for d in (5, 6, 7):
        md_mask = df['max_day'] == d
        ind[f'total_day_{d}_euploid']   = (md_mask & has_euploid  ).astype(int)
        ind[f'total_day_{d}_aneuploid'] = (md_mask & has_aneuploid).astype(int)

    # Grade-specific euploid / aneuploid per max_day
    for d in (5, 6, 7):
        md_mask = df['max_day'] == d
        for g in BLAST_GRADES:
            gm = grade_up == g
            ind[f'total_day{d}_euploid_{g}']   = (md_mask & gm & has_euploid  ).astype(int)
            ind[f'total_day{d}_aneuploid_{g}'] = (md_mask & gm & has_aneuploid).astype(int)

    # Overall PGD-result counts
    ind['total_complex_aneuploid'] = has_complex_aneu.astype(int)
    ind['total_low_mosaic']        = has_low_mosaic.astype(int)
    ind['total_high_mosaic']       = has_high_mosaic.astype(int)
    ind['total_chaotic']           = has_chaotic.astype(int)
    ind['total_euploid']           = has_euploid.astype(int)
    ind['total_aneuploid']         = (has_aneuploid & ~has_complex_aneu).astype(int)
    ind['total_untested']          = is_untested.astype(int)
    ind['total_blastocysts']       = is_blast.astype(int)

    # ---- Aggregate per (patid, cycleid) ----
    agg_map = {c: 'sum' for c in ind.columns if c not in ('patid', 'cycleid', '__ord__')}
    agg_map['__ord__'] = 'max'

    wide = ind.groupby(['patid', 'cycleid'], as_index=False, dropna=False).agg(agg_map)
    wide = wide.rename(columns={'__ord__': 'total_eggs_ICSId'})

    count_cols = [c for c in wide.columns if c not in ('patid', 'cycleid')]
    wide[count_cols] = wide[count_cols].fillna(0).astype(int)

    # ---- Determine cycle type ----
    stage_up = (
        df["stage"]
        .astype("string")
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
    )

    OOCYTE_FREEZE_STAGES = {
        "M1",
        "M2",
        "GV",
        "EMPTYZONA",
        "DEGENERATED"
    }

    def classify_cycle(stages):
        unique_stages = set(stages.dropna())

        if len(unique_stages) > 0 and unique_stages.issubset(OOCYTE_FREEZE_STAGES):
            return "oocyte freeze"
        else:
            return "IVF"

    cycle_type = (
        pd.DataFrame({
            "patid": df["patid"],
            "cycleid": df["cycleid"],
            "stage": stage_up
        })
        .groupby(["patid", "cycleid"], dropna=False)["stage"]
        .apply(classify_cycle)
        .reset_index(name="cycle_type")
    )

    wide = wide.merge(
        cycle_type,
        on=["patid", "cycleid"],
        how="left"
    )

    return wide

In [ ]:
########################################################################################################################################################
# Cleaning needle type
########################################################################################################################################################

def clean_needle_type(x):
    if pd.isna(x):
        return pd.NA

    if x in [
        "single",
        "single lumen",
        "single lumen needle",
        "cook echotip 16 g"
    ]:
        return "single lumen"

    elif x in [
        "double lumen",
        "double lumen needle"
    ]:
        return "double lumen"

    elif x in [
        "cook echotip 16 g, double lumen",
        "cook echotip 16 g, double",
        "cook echotip 16 g, double lumen needle",
        "double lumen needle and single lumen",
        "cook echotip 16 g,, double lumen needle",
        "cook echotip 16 g, double lumen",
        "cook echotip 16 g, double",
        "double lumen needle, cook echotip 16 g"
    ]:
        return "single and double lumen"

    else:
        return x

In [ ]:
"""
########################################################################################################################################################
# In emr_cycle, I want to separate details of each cycle
########################################################################################################################################################

def organize_cycle_type(x):
    
    #Parse a raw cycle_name string into structured fields.

    #Returns:
    #    clinic, cycle_type, donor_type,
    #    protocol_family, protocol,
    #    conversion, conversion_targets, original_cycle,
    #    freeze_all, classification_status
    

    # ---------------------------------------------------------
    # MISSING VALUES
    # ---------------------------------------------------------
    if pd.isna(x):
        return {
            "clinic": pd.NA, "cycle_type": pd.NA, "donor_type": pd.NA,
            "protocol_family": pd.NA, "protocol": pd.NA,
            "conversion": pd.NA, "conversion_targets": pd.NA,
            "original_cycle": pd.NA, "freeze_all": pd.NA,
            "classification_status": "REVIEW",
        }

    raw = str(x)
    s = raw.upper().strip()

    # ---------------------------------------------------------
    # NORMALIZATION - typo fixes + abbreviation expansion
    # ---------------------------------------------------------
    s = s.replace("OOYTE", "OOCYTE").replace("OOCTYE", "OOCYTE")
    s = s.replace("KASIER", "KAISER").replace("AUTOLOGUS", "AUTOLOGOUS")
    s = s.replace("→", "->").replace("-->", "->")

    s = re.sub(r'\bLET\b',   'LETROZOLE',  s)
    s = re.sub(r'\bNAT\b',   'NATURAL',    s)
    s = re.sub(r'\bANT\b',   'ANTAGONIST', s)
    s = re.sub(r'\bANTAG\b', 'ANTAGONIST', s)
    s = re.sub(r'\bADJ\b',   'ADJUVANTS',  s)

    # Collapse "BCP + Antagonist" wording into "BCPA" up front
    # (BCPA = BCP + Antagonist by definition)
    if re.search(r'\bBCP\b', s) and "ANTAGONIST" in s:
        s = re.sub(r'\bBCP\b\s*ANTAGONIST', 'BCPA', s)
        s = re.sub(r'\bBCP\b(?=.*ANTAGONIST)', 'BCPA', s)  # any remaining BCP -> BCPA

    # ---------------------------------------------------------
    # CLINIC
    # ---------------------------------------------------------
    clinic = "Kaiser" if "KAISER" in s else "RPSD"

    # ---------------------------------------------------------
    # DONOR / THIRD-PARTY TYPE
    # ---------------------------------------------------------
    if "RECIPROCAL" in s:
        donor_type = "Reciprocal (same-sex partners)"
    elif "DONOR EMBRYO" in s:
        donor_type = "Donor Embryo Recipient"
    elif "EGG DONOR" in s or "EGG DONATION" in s or "DONOR OOCYTE" in s:
        donor_type = "Egg Donor"
    elif "GC FET" in s or "GESTATIONAL CARRIER" in s:
        donor_type = "Gestational Carrier"
    elif "COMPASSIONATE" in s:
        donor_type = "Compassionate Transfer"
    elif "THIRD PARTY" in s and re.search(r'\bIP\b', s):
        donor_type = "Intended Parent (third party)"
    elif "THIRD PARTY" in s:
        donor_type = "Third Party"
    elif "INTENDED PARENT" in s:
        donor_type = "Intended Parent"
    elif "RECIPIENT" in s:
        donor_type = "Recipient"
    elif "DONOR CYCLE" in s or re.search(r'\bDONOR\b', s):
        donor_type = "Donor (unspecified)"
    else:
        donor_type = "Autologous"

    # ---------------------------------------------------------
    # CYCLE TYPE  (base only — no conversion arrows)
    # ---------------------------------------------------------
    if "SPONTANEOUS PREGNANCY" in s:
        cycle_type = "Spontaneous Pregnancy"
    elif "MOCK CYCLE" in s or "TEST CYCLE" in s:
        cycle_type = "Mock Cycle"
    elif "IVF" in s and "FET" in s and "CONVERTED" not in s and "CONVERT" not in s:
        cycle_type = "IVF & FET"
    elif "IVF" in s and "FRESH ET" in s:
        cycle_type = "IVF & Fresh ET"
    elif "FRESH ET" in s:
        cycle_type = "Fresh ET"
    elif (any(term in s for term in [
                "OOCYTE FREEZE", "OOCYTE: FREEZE", "OOCYTE AUTOLOGOUS FREEZE",
                "OOCYTE FREEZE ALL", "EGG FREEZE",
            ])
          or s.startswith("OV:")
          or "OV MLEA" in s or "OV SPLIT" in s or "IVF/ OV SPLIT" in s):
        cycle_type = "Oocyte Freeze"
    elif "OOCYTE THAW" in s:
        cycle_type = "Oocyte Thaw"
    elif any(term in s for term in [
                "EMBRYO THAW", "EMBRYO FREEZE",
                "EMBRYO RE-FREEZE", "EMBRYO REFREEZE",
            ]):
        cycle_type = "Embryo Thaw"
    elif "IVF" in s:
        cycle_type = "IVF"
    elif "FET" in s or "FROZEN EMBRYO TRANSFER" in s:
        cycle_type = "FET"
    elif "TDI" in s:
        cycle_type = "TDI"
    elif "IUI" in s:
        cycle_type = "IUI"
    elif "TIC" in s:
        cycle_type = "TIC"
    elif s in ("MLEA", "OV MLEA SS + CC/ NO ADJUVANTS"):
        cycle_type = "Oocyte Freeze"
    else:
        cycle_type = "Unknown"

    # ---------------------------------------------------------
    # FREEZE ALL
    # ---------------------------------------------------------
    freeze_all = any(k in s for k in ("FREEZE ALL", "FREEZE - ALL", "FREEZE-ALL"))

    # ---------------------------------------------------------
    # CONVERSION DETECTION  (belongs to PROTOCOL, not cycle_type)
    # ---------------------------------------------------------
    conversion_targets = []

    def any_of(patterns):
        return any(p in s for p in patterns)

    if any_of(["CONVERTED TO IUI", "CONVERT TO IUI", "CONVERTED IUI",
               "CONVERTING TO IUI", "-> IUI"]):
        conversion_targets.append("IUI")
    if any_of(["CONVERTED TO TIC", "CONVERT TO TIC", "CONVERTED TIC",
               "CONVERTING TO TIC", "-> TIC"]):
        conversion_targets.append("TIC")
    if any_of(["CONVERTED TO IVF", "CONVERT TO IVF", "CONVERTED IVF",
               "CONVERTING TO IVF", "-> IVF"]):
        conversion_targets.append("IVF")
    if any_of(["CONVERTED TO MED FET", "CONVERTED TO MEDICATED FET",
               "-> MED FET", "-> MEDICATED FET",
               "CONVERTED TO MEDICATED", "CONVERTED TO MED"]):
        conversion_targets.append("Medicated FET")
    elif any_of(["CONVERTED TO FET", "CONVERT TO FET", "CONVERTED FET",
                 "CONVERTING TO FET", "-> FET"]):
        conversion_targets.append("FET")
    if any_of(["CONVERTED TO COH", "CONVERT TO COH", "-> COH",
               "CONVERTING TO COH"]):
        conversion_targets.append("COH")
    if any_of(["CONVERTED TO LETROZOLE", "-> LETROZOLE"]):
        conversion_targets.append("Letrozole")
    if any_of(["CONVERTED TO NATURAL", "-> NATURAL"]):
        conversion_targets.append("Natural")
    if any_of(["CONVERTED TO PATCHES", "CONVERTED TO TRANSDERMAL",
               "-> PATCHES", "-> TRANSDERMAL"]):
        conversion_targets.append("Transdermal Patches")
    if any_of(["CONVERTED TO VAGINAL ESTRACE", "-> VAGINAL ESTRACE"]):
        conversion_targets.append("Vaginal Estrace")
    if any_of(["CONVERTED TO ORAL ESTRACE", "CONVERTED TO ESTRACE 2MG",
               "-> ORAL ESTRACE", "-> ESTRACE 2MG"]):
        conversion_targets.append("Oral Estrace")
    if any_of(["CONVERTED TO BCP"]):
        conversion_targets.append("BCP")
    if any_of(["CONVERTED TO EGG FREEZE", "CONVERTED TO OOCYTE FREEZE",
               "CONVERTED TO EGG FREEZING"]):
        conversion_targets.append("Oocyte Freeze")

    conversion_targets = list(dict.fromkeys(conversion_targets))
    conversion = len(conversion_targets) > 0

    # "converted from X" context (does NOT flag conversion)
    original_cycle = None
    if "CONVERTED FROM CC IUI" in s or "FROM CC IUI" in s:
        original_cycle = "CC IUI"
    elif "CONVERTED FROM IVF" in s or ("FROM IVF" in s and "CONVERTED" in s):
        original_cycle = "IVF"
    elif "CONVERTED FROM IUI" in s or ("FROM IUI" in s and "CONVERTED" in s):
        original_cycle = "IUI"

    # ---------------------------------------------------------
    # PROTOCOL FAMILY + PROTOCOL (detailed)
    # ---------------------------------------------------------
    protocol_family = pd.NA
    protocol = pd.NA

    # ---- Thaw cycles: dispatch first ----
    if cycle_type == "Oocyte Thaw":
        parts = []
        if "ICSI" in s:                            parts.append("ICSI")
        if "BIOPSY" in s:                          parts.append("Biopsy")
        if "RE-FREEZE" in s or "REFREEZE" in s:    parts.append("Re-Freeze")
        if "NO PGTA" in s:                         parts.append("No PGTa")
        protocol = " / ".join(parts) if parts else "Oocyte Thaw"
        protocol_family = "Oocyte Thaw"

    elif cycle_type == "Embryo Thaw":
        parts = []
        if "RE-BIOPSY" in s:                       parts.append("Re-Biopsy")
        elif "BIOPSY" in s:                        parts.append("Biopsy")
        if "ICSI" in s:                            parts.append("ICSI")
        if "RE-FREEZE" in s or "REFREEZE" in s:    parts.append("Re-Freeze")
        protocol = " / ".join(parts) if parts else "Embryo Thaw"
        protocol_family = "Embryo Thaw"

    else:
        # ---- BCPA family (before BCP because BCPA contains "BCP") ----
        # "BCP Antagonist" / "BCP ANT" was normalized to "BCPA" up top.
        if "BCPA" in s and "LUPRON OVERLAP" in s:
            protocol_family = "BCPA"
            if "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "BCPA Lupron Overlap Gonadotropins +/- Clomid +/- Adjuvants"
            elif "ADJUVANTS" in s:
                protocol = "BCPA Lupron Overlap Gonadotropins +/- Adjuvants"
            else:
                protocol = "BCPA Lupron Overlap"

        elif "BCPA" in s:
            protocol_family = "BCPA"
            if "GONADOTROPINS" in s and "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "BCPA Gonadotropins +/- Clomid +/- Adjuvants"
            elif "GONADOTROPINS" in s and "ADJUVANTS" in s:
                protocol = "BCPA Gonadotropins +/- Adjuvants"
            elif "GONADOTROPINS" in s:
                protocol = "BCPA Gonadotropins"
            elif "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "BCPA +/- Clomid +/- Adjuvants"
            else:
                protocol = "BCPA"

        # ---- BCP family (plain BCP, no antagonist) ----
        elif re.search(r'\bBCP\b', s) and "LUPRON OVERLAP" in s:
            protocol_family = "BCP"
            if "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "BCP Lupron Overlap Gonadotropins +/- Clomid +/- Adjuvants"
            elif "ADJUVANTS" in s:
                protocol = "BCP Lupron Overlap Gonadotropins +/- Adjuvants"
            else:
                protocol = "BCP Lupron Overlap"

        elif re.search(r'\bBCP\b', s):
            protocol_family = "BCP"
            if "GONADOTROPINS" in s and "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "BCP Gonadotropins +/- Clomid +/- Adjuvants"
            elif "GONADOTROPINS" in s and "ADJUVANTS" in s:
                protocol = "BCP Gonadotropins +/- Adjuvants"
            elif "GONADOTROPINS" in s:
                protocol = "BCP Gonadotropins"
            elif "NO ADJUVANTS" in s:
                protocol = "BCP No Adjuvants"
            elif "RANDOM" in s:
                protocol = "BCP Random Start"
            else:
                protocol = "BCP"

        # ---- MLEA family ----
        elif "MLEA" in s and "LUPRON OVERLAP" in s:
            protocol_family = "MLEA"
            if "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "MLEA Lupron Overlap Gonadotropins +/- Clomid +/- Adjuvants"
            elif "ADJUVANTS" in s:
                protocol = "MLEA Lupron Overlap Gonadotropins +/- Adjuvants"
            else:
                protocol = "MLEA Lupron Overlap"

        elif "MLEA" in s and ("RANDOM" in s or re.search(r'\bSS\b', s)):
            protocol_family = "MLEA"
            protocol = "MLEA Random Stim Start"

        elif "MLEA" in s and "LOW STIM" in s:
            protocol_family = "MLEA"
            protocol = "MLEA Low Stim"

        elif "MLEA" in s:
            protocol_family = "MLEA"
            if "GONADOTROPINS" in s and "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "MLEA Gonadotropins +/- Clomid +/- Adjuvants"
            elif "GONADOTROPINS" in s and "ADJUVANTS" in s:
                protocol = "MLEA Gonadotropins +/- Adjuvants"
            elif "GONADOTROPINS" in s:
                protocol = "MLEA Gonadotropins"
            elif "ADJUVANTS" in s:
                protocol = "MLEA +/- Adjuvants"
            else:
                protocol = "MLEA"

        # ---- Long Estrace ----
        elif "LONG ESTRACE" in s:
            protocol_family = "Long Estrace"
            if "GONADOTROPINS" in s and "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "Long Estrace Gonadotropins +/- Clomid +/- Adjuvants"
            elif "GONADOTROPINS" in s and "ADJUVANTS" in s:
                protocol = "Long Estrace Gonadotropins +/- Adjuvants"
            elif "GONADOTROPINS" in s:
                protocol = "Long Estrace Gonadotropins"
            elif "CLOMID" in s and "ADJUVANTS" in s:
                protocol = "Long Estrace +/- Clomid +/- Adjuvants"
            else:
                protocol = "Long Estrace"

        # ---- Stim-start protocols ----
        elif "ONCO" in s and "RANDOM" in s:
            protocol_family = "Random Stim Start"
            protocol = "ONCO Random Stim Start"
        elif "RANDOM STIM START" in s or "RANDOM START" in s:
            protocol_family = "Random Stim Start"
            protocol = "Random Stim Start"
        elif "FOLLICULAR PHASE START" in s:
            protocol_family = "Follicular Phase Start"
            protocol = "Follicular Phase Start"
        elif "CD3 STIM START" in s:
            protocol_family = "CD3 Stim Start"
            protocol = "CD3 Stim Start"
        elif "MIDLUTEAL" in s or "MID-LUTEAL" in s or "MID LUTEAL" in s or "LH+7" in s:
            protocol_family = "MidLuteal Stim Start"
            protocol = "MidLuteal Stim Start"

        # ---- Other named IVF protocols ----
        elif "GANIRELIX" in s:
            protocol_family = "Ganirelix"
            protocol = "Ganirelix"
        elif "ANTAGONIST" in s:
            # Antagonist without BCP context — standalone antagonist protocol
            protocol_family = "Antagonist"
            protocol = "Antagonist"
        elif "AMH IVF" in s:
            protocol_family = "AMH"
            protocol = "AMH"
        elif "FALLUM" in s:
            protocol_family = "Fallum"
            protocol = "Fallum"
        elif "MENOPUR" in s and cycle_type not in ("IUI", "TIC", "TDI"):
            protocol_family = "Menopur"
            protocol = "Menopur"
        elif "LOW STIM" in s:
            protocol_family = "Low Stim"
            protocol = "Low Stim"
        elif "CUSTOM SCHEDULE" in s:
            protocol_family = "Custom Schedule"
            protocol = "Custom Schedule"

        # ---- FET-side protocols ----
        elif cycle_type in ("FET", "IVF & FET", "Mock Cycle", "Fresh ET", "IVF & Fresh ET"):
            fet_parts = []
            if "UNSTIM" in s:
                fet_parts.append("Unstimulated")
            if "TRANSDERMAL PATCH" in s or "PATCHES" in s or re.search(r'\bPATCH\b', s):
                fet_parts.append("Transdermal Patches")
            if "VAGINAL ESTRACE" in s:
                fet_parts.append("Vaginal Estrace")
            if "ORAL ESTRACE" in s or "ESTRACE ORAL" in s or "ESTRACE 2MG" in s:
                fet_parts.append("Oral Estrace")
            if "ESTRADIOL VALERATE" in s:
                fet_parts.append("Estradiol Valerate")
            if "COH" in s:
                fet_parts.append("COH")
            mod_nat = ("MODIFIED NATURAL" in s or "MOD NATURAL" in s or "MOD LETROZOLE" in s)
            if mod_nat:
                fet_parts.append("Modified Natural")
            if "LETROZOLE" in s:
                fet_parts.append("Letrozole")
            if "NATURAL" in s and not mod_nat:
                fet_parts.append("Natural")

            fet_parts = list(dict.fromkeys(fet_parts))
            if fet_parts:
                protocol = " + ".join(fet_parts) if len(fet_parts) > 1 else fet_parts[0]
                if any(p in fet_parts for p in [
                    "Transdermal Patches", "Vaginal Estrace",
                    "Oral Estrace", "Estradiol Valerate",
                ]):
                    protocol_family = "Medicated FET"
                elif "COH" in fet_parts:
                    protocol_family = "COH FET"
                elif "Modified Natural" in fet_parts:
                    protocol_family = "Modified Natural FET"
                elif "Letrozole" in fet_parts:
                    protocol_family = "Letrozole FET"
                elif "Natural" in fet_parts:
                    protocol_family = "Natural FET"
                elif "Unstimulated" in fet_parts:
                    protocol_family = "Unstimulated"

        # ---- IUI / TDI / TIC-side protocols ----
        elif cycle_type in ("IUI", "TDI", "TIC"):
            if "LETROZOLE" in s and "CLOMID" in s:
                protocol_family = "Letrozole + Clomid"
                protocol = "Letrozole + Clomid"
            elif "LETROZOLE" in s:
                protocol_family = "Letrozole"
                protocol = "Letrozole"
            elif "CLOMID" in s:
                protocol_family = "Clomid"
                protocol = "Clomid"
            elif "COH" in s:
                protocol_family = "COH"
                protocol = "COH"
            elif "MENOPUR" in s:
                protocol_family = "Menopur"
                protocol = "Menopur"
            elif "NATURAL" in s:
                protocol_family = "Natural"
                protocol = "Natural"

    # ---------------------------------------------------------
    # APPEND CONVERSION TO PROTOCOL (not cycle_type)
    # ---------------------------------------------------------
    if conversion_targets:
        conv_str = " / ".join(conversion_targets)
        if pd.isna(protocol):
            protocol = f"→ {conv_str}"
        else:
            protocol = f"{protocol} → {conv_str}"

    # ---------------------------------------------------------
    # CLASSIFICATION STATUS
    # ---------------------------------------------------------
    if cycle_type == "Unknown":
        classification_status = "REVIEW"
    elif (pd.isna(protocol) and cycle_type not in (
            "Spontaneous Pregnancy", "Mock Cycle", "Oocyte Thaw", "Embryo Thaw")):
        classification_status = "REVIEW"
    else:
        classification_status = "OK"

    return {
        "clinic": clinic,
        "cycle_type": cycle_type,
        "donor_type": donor_type,
        "protocol_family": protocol_family,
        "protocol": protocol,
        "conversion": conversion,
        "conversion_targets": conversion_targets if conversion_targets else pd.NA,
        "original_cycle": original_cycle,
        "freeze_all": freeze_all,
        "classification_status": classification_status,
    }
"""

In [ ]:
########################################################################################################################################################
# In emr_cycle, I want to separate details of each cycle
########################################################################################################################################################

# =============================================================================
# UUID LOOKUP TABLES
# =============================================================================
# These are HAND-CODED after inspecting the full value_counts per UUID.
# Verify each entry by running:
#     for uuid in merged_df['plan_treatment'].value_counts().index:
#         grp = merged_df[merged_df['plan_treatment'] == uuid]
#         print(f"\n=== {uuid} (n={len(grp)}) ===")
#         print(grp['cycle_name'].value_counts().head(10))
# Same loop for 'cycletype'. Adjust labels below where inference was wrong.

PLAN_TREATMENT_MAP = {
    'f7a68afe-e5b6-4a60-8a4b-a6ca7ada44a5': {'cycle_type': 'FET',            'clinic': 'RPSD',   'donor_type': 'Autologous'},
    '4e625189-d13b-4ca1-9f4d-463506ae7a5f': {'cycle_type': 'IVF Freeze-All', 'clinic': 'RPSD',   'donor_type': 'Autologous'},
    '31874673-6e93-4ec2-b175-a8e2cc4860af': {'cycle_type': 'IUI',            'clinic': 'RPSD',   'donor_type': 'Autologous'},
    '02672e1f-371b-433a-be83-8664dfe7412d': {'cycle_type': 'Oocyte Freeze',  'clinic': 'RPSD',   'donor_type': 'Autologous'},
    'ab78f475-0fd0-450e-ab20-067dc3220cfe': {'cycle_type': 'IVF Freeze-All', 'clinic': 'Kaiser', 'donor_type': 'Autologous'},
    '5448a2c2-7f1b-4e17-8b75-c06cbd27f139': {'cycle_type': 'FET',            'clinic': 'Kaiser', 'donor_type': 'Autologous'},
    'fa31fcbc-4673-48e0-b343-142624a43558': {'cycle_type': 'TIC',            'clinic': 'RPSD',   'donor_type': 'Autologous'},
    '921975ec-5e3c-4f40-92fa-2d15c316a199': {'cycle_type': 'FET',            'clinic': 'RPSD',   'donor_type': 'Reciprocal / Third-party IP'},
    'c9c3d726-51c4-4777-9e45-47743ed81c3e': {'cycle_type': 'FET',            'clinic': 'RPSD',   'donor_type': 'Gestational Carrier'},
    '2e37cde6-a522-46a5-bcdd-060965c7a932': {'cycle_type': 'Oocyte Thaw',    'clinic': 'RPSD',   'donor_type': 'Autologous'},
    'b96bf6c4-dfe5-4ec8-a5f2-fefb1c24f241': {'cycle_type': 'Embryo Thaw',    'clinic': 'RPSD',   'donor_type': 'Autologous'},
    '52450634-c3d6-44ee-84c6-a39ccf9a4643': {'cycle_type': 'Oocyte Freeze',  'clinic': 'Kaiser', 'donor_type': 'Autologous'},
    '43112291-203a-472a-82cc-4a644af0e714': {'cycle_type': 'IVF Freeze-All', 'clinic': 'RPSD',   'donor_type': 'Intended Parent (third party)'},
    'f4344aaa-46a1-4c3f-ab3a-770d0dcf5394': {'cycle_type': 'IVF Fresh ET',   'clinic': 'RPSD',   'donor_type': 'Recipient (third party)'},
    'dd0d444a-43cd-4808-9e35-0a9620ccdb8c': {'cycle_type': 'IVF Freeze-All', 'clinic': 'RPSD',   'donor_type': 'Egg Donor'},
    '11a21c89-b586-473f-b250-84df4fc9cf13': {'cycle_type': 'Oocyte Thaw → Fresh ET', 'clinic': 'RPSD', 'donor_type': 'Autologous'},
    'd4c5cbb4-3de9-44bb-b01e-557ab61483e3': {'cycle_type': 'Spontaneous Pregnancy', 'clinic': 'RPSD', 'donor_type': 'Autologous'},
    'edbdec87-9a12-47db-b7a1-dc38385c419e': {'cycle_type': 'IVF (named protocol)',  'clinic': 'RPSD', 'donor_type': 'Autologous'},
    'bd973b25-9d29-44f4-8691-a083fcf496ba': {'cycle_type': 'Oocyte Thaw',    'clinic': 'Kaiser', 'donor_type': 'Autologous'},
    '1742ecc2-f062-4947-b9b1-48a8a5745867': {'cycle_type': 'Mock Cycle',     'clinic': 'RPSD',   'donor_type': 'Autologous'},
    'f1436db2-46a6-45cf-ae9b-f524c526a70a': {'cycle_type': 'IVF Fresh ET',   'clinic': 'RPSD',   'donor_type': 'Recipient / GC (third party)'},
    '28f5d566-df7f-4b51-ad5a-9b8a87d1dfdb': {'cycle_type': 'Oocyte Freeze',  'clinic': 'RPSD',   'donor_type': 'Intended Parent (third party)'},
    'a032b62e-2ed9-47b9-8afc-49bedc2092ac': {'cycle_type': 'Embryo Freeze',  'clinic': 'RPSD',   'donor_type': 'Autologous'},
}

CYCLETYPE_MAP = {
    '5cf37237-dd8c-4bcc-ae5e-3da7717a21cb': {'protocol_family': 'MLEA'},
    'ac36dd51-7809-4975-83de-48cda51987f5': {'protocol_family': 'BCPA'},
    '4625ff0d-e082-45f9-a814-3267fc70b647': {'protocol_family': 'BCP Lupron Overlap'},
    '9a4aac0e-cf54-458a-87dc-506edb0e98ec': {'protocol_family': 'Long Estrace'},
    'ae7c0905-13b6-41a9-899e-fe7f883bf28f': {'protocol_family': 'Random Stim Start'},
    '31874673-6e93-4ec2-b175-a8e2cc4860af': {'protocol_family': 'IUI'},
    '02c9013f-39e7-4140-9f11-93da2ec9ec69': {'protocol_family': 'IUI: Clomid'},
    'c6dfc626-dbfd-4b4c-9458-aba920686d63': {'protocol_family': 'IUI: Letrozole'},
    '9160e0b6-61f3-4fab-9844-356d49f695b9': {'protocol_family': 'IUI/TDI: Natural'},
    '56d82a3e-1d73-4ddb-a1a2-3b691e0d771d': {'protocol_family': 'TIC: Clomid'},
    '565c2998-c355-4eef-8852-5d5cbb608750': {'protocol_family': 'TIC: Letrozole'},
    'e772bdbc-a06d-4d95-8871-9fe50ebdd768': {'protocol_family': 'CC IUI'},
    '3e0aacef-70d9-40d3-81a9-b2d0494a8c20': {'protocol_family': 'FET: Transdermal Patches'},
    '7bb2f39f-bb02-4a72-8c82-4fb9538fe83a': {'protocol_family': 'FET: Natural'},
    '2a5fe645-02d7-4635-9646-09785a1795ab': {'protocol_family': 'FET: COH'},
    '947b4bed-52b4-44e4-a5b6-d1bbbaee9624': {'protocol_family': 'FET: Vaginal Estrace'},
    'f8be9759-4357-446f-8bd6-3f2a90c3b8c9': {'protocol_family': 'FET: Oral Estrace'},
    '8144853b-4e29-4f56-9f6b-3ef9e19cc99c': {'protocol_family': 'FET: Estradiol Valerate'},
    '74b492c8-0b79-4326-808a-162957ef80db': {'protocol_family': 'FET: BCP Lupron Overlap + Patches'},
    'bcba2438-64ce-43b7-acab-11be28ab09ca': {'protocol_family': 'AMH IVF'},
    'c0e35ad1-53bd-4e80-a8b0-1f0bb241c292': {'protocol_family': 'Fallum IVF'},
    '87e15df5-2f3a-45e6-8a87-cc2957150773': {'protocol_family': 'Ganirelix IVF'},
    'a6fc6fd8-a45d-45f6-b44a-5d9698376cfa': {'protocol_family': 'Antagonist IVF'},
    'dd14df42-56a5-47ec-9333-d96cbf526c63': {'protocol_family': 'BCP Lupron Overlap Fresh ET'},
    'bda9edb6-f917-4e93-9c60-7aaca42072d2': {'protocol_family': 'BCP'},
    'ab9d5748-0cf9-4e66-9bfc-49be0b0560ae': {'protocol_family': 'BCP'},
    '1549a58f-4627-428d-919f-46e3fa6b508b': {'protocol_family': 'BCPA Fresh ET'},
    'be20a948-3891-4997-b4cf-e8e9dfed925b': {'protocol_family': 'Custom Schedule'},
    'fbc63f5b-7ee3-4485-b980-c4a8fa374ed1': {'protocol_family': 'MLEA (no Clomid)'},
    'acd8e071-e22a-4459-8587-6e1318d72e13': {'protocol_family': 'IVF Test'},
    'e6bebbff-9768-4745-a7ef-2aef06f5ec3e': {'protocol_family': 'Donor Test / MLEA'},
    '5c0a9cdd-db26-4979-9454-95010a74bd94': {'protocol_family': 'Egg Donation'},
    '12f6e8cc-ebae-4af1-87de-88b31cb3dc6a': {'protocol_family': 'IUI: COH'},
}


# =============================================================================
# REGEX PARSER
# =============================================================================
# Only extracts what the UUIDs don't give us: conversions, adjuvants,
# freeze_all flag, stim_start variant, and "converted from X" context.
# Also provides a fallback protocol_family if cycletype UUID is missing.

def parse_cycle_name(cycle_name):
    """
    Parse free-text cycle_name for fields the UUID lookups don't cover.

    Returns dict with:
        conversion, conversion_targets, original_cycle,
        adjuvants (list), freeze_all (bool), unstimulated (bool),
        stim_start_variant, protocol_family_regex (fallback only)
    """
    result = {
            'conversion': False,
            'conversion_targets': None,
            'original_cycle': None,
            'adjuvants': None,
            'freeze_all': False,
            'unstimulated': False,
            'stim_start_variant': None,
            'protocol_family_regex': None,
            # Cycle purpose flags
            'is_ivf': False, 'is_fet': False, 'is_iui': False, 'is_tic': False,
            'is_tdi': False, 'is_oocyte_freeze': False, 'is_oocyte_thaw': False,
            'is_embryo_thaw': False, 'is_embryo_freeze': False,
            'is_mock': False, 'is_spontaneous': False,
            # Donor / third-party flags
            'is_third_party': False, 'is_intended_parent': False,
            'is_recipient': False, 'is_gc': False, 'is_egg_donor': False,
            'is_donor_embryo': False, 'is_reciprocal': False,
            'is_compassionate': False, 'is_autologous': False,
            # Clinic
            'is_kaiser': False,
            # Base protocol + modifiers
            'protocol_base': None,
            'has_lupron_overlap': False, 'has_gonadotropins': False,
            'has_clomid': False, 'has_letrozole': False, 'has_menopur': False,
            'has_low_stim': False, 'adjuvants_flag': None,
            # FET medication
            'fet_medication': None,
        }

    if pd.isna(cycle_name):
        return result

    raw = str(cycle_name)
    s = raw.upper().strip()

    # ---- Normalization ----
    s = s.replace("OOYTE", "OOCYTE").replace("OOCTYE", "OOCYTE")
    s = s.replace("KASIER", "KAISER").replace("AUTOLOGUS", "AUTOLOGOUS")
    s = s.replace("→", "->").replace("-->", "->")

    s = re.sub(r'\bLET\b',   'LETROZOLE',  s)
    s = re.sub(r'\bNAT\b',   'NATURAL',    s)
    s = re.sub(r'\bANT\b',   'ANTAGONIST', s)
    s = re.sub(r'\bANTAG\b', 'ANTAGONIST', s)
    s = re.sub(r'\bADJ\b',   'ADJUVANTS',  s)

    # "BCP Antagonist" -> "BCPA" (they mean the same thing)
    if re.search(r'\bBCP\b', s) and "ANTAGONIST" in s:
        s = re.sub(r'\bBCP\b\s*ANTAGONIST', 'BCPA', s)
        s = re.sub(r'\bBCP\b(?=.*ANTAGONIST)', 'BCPA', s)

    # ---- FREEZE ALL ----
    result['freeze_all'] = any(k in s for k in (
        "FREEZE ALL", "FREEZE - ALL", "FREEZE-ALL"
    ))

    # ---- UNSTIM DETECTION ----
    result['unstimulated'] = "UNSTIM" in s

    # ---- CONVERSION DETECTION ----
    conversion_targets = []

    def any_of(patterns):
        return any(p in s for p in patterns)

    if any_of(["CONVERTED TO IUI", "CONVERT TO IUI", "CONVERTED IUI",
               "CONVERTING TO IUI", "-> IUI"]):
        conversion_targets.append("IUI")
    if any_of(["CONVERTED TO TIC", "CONVERT TO TIC", "CONVERTED TIC",
               "CONVERTING TO TIC", "-> TIC"]):
        conversion_targets.append("TIC")
    if any_of(["CONVERTED TO IVF", "CONVERT TO IVF", "CONVERTED IVF",
               "CONVERTING TO IVF", "-> IVF"]):
        conversion_targets.append("IVF")
    if any_of(["CONVERTED TO MED FET", "CONVERTED TO MEDICATED FET",
               "-> MED FET", "-> MEDICATED FET",
               "CONVERTED TO MEDICATED", "CONVERTED TO MED"]):
        conversion_targets.append("Medicated FET")
    elif any_of(["CONVERTED TO FET", "CONVERT TO FET", "CONVERTED FET",
                 "CONVERTING TO FET", "-> FET"]):
        conversion_targets.append("FET")
    if any_of(["CONVERTED TO COH", "CONVERT TO COH", "-> COH",
               "CONVERTING TO COH"]):
        conversion_targets.append("COH")
    if any_of(["CONVERTED TO LETROZOLE", "-> LETROZOLE"]):
        conversion_targets.append("Letrozole")
    if any_of(["CONVERTED TO NATURAL", "-> NATURAL"]):
        conversion_targets.append("Natural")
    if any_of(["CONVERTED TO PATCHES", "CONVERTED TO TRANSDERMAL",
               "-> PATCHES", "-> TRANSDERMAL"]):
        conversion_targets.append("Transdermal Patches")
    if any_of(["CONVERTED TO VAGINAL ESTRACE", "-> VAGINAL ESTRACE"]):
        conversion_targets.append("Vaginal Estrace")
    if any_of(["CONVERTED TO ORAL ESTRACE", "CONVERTED TO ESTRACE 2MG",
               "-> ORAL ESTRACE", "-> ESTRACE 2MG"]):
        conversion_targets.append("Oral Estrace")
    if any_of(["CONVERTED TO BCP"]):
        conversion_targets.append("BCP")
    if any_of(["CONVERTED TO EGG FREEZE", "CONVERTED TO OOCYTE FREEZE",
               "CONVERTED TO EGG FREEZING"]):
        conversion_targets.append("Oocyte Freeze")

    conversion_targets = list(dict.fromkeys(conversion_targets))
    result['conversion'] = len(conversion_targets) > 0
    result['conversion_targets'] = conversion_targets if conversion_targets else None

    # "converted from X" context
    if "CONVERTED FROM CC IUI" in s or "FROM CC IUI" in s:
        result['original_cycle'] = "CC IUI"
    elif "CONVERTED FROM IVF" in s or ("FROM IVF" in s and "CONVERTED" in s):
        result['original_cycle'] = "IVF"
    elif "CONVERTED FROM IUI" in s or ("FROM IUI" in s and "CONVERTED" in s):
        result['original_cycle'] = "IUI"

    # ---- ADJUVANTS / DRUG COMBOS ----
    adjuvants = []
    if "GONADOTROPINS" in s:                             adjuvants.append("Gonadotropins")
    if "CLOMID" in s:                                    adjuvants.append("Clomid")
    if "LUPRON OVERLAP" in s:                            adjuvants.append("Lupron Overlap")
    if "NO ADJUVANTS" in s:                              adjuvants.append("No Adjuvants")
    elif "ADJUVANTS" in s:                               adjuvants.append("+/- Adjuvants")
    if "MENOPUR" in s:                                   adjuvants.append("Menopur")
    result['adjuvants'] = adjuvants if adjuvants else None

    # ---- STIM START VARIANT ----
    if "ONCO" in s and "RANDOM" in s:
        result['stim_start_variant'] = "ONCO Random"
    elif "RANDOM STIM START" in s or "RANDOM START" in s:
        result['stim_start_variant'] = "Random"
    elif "FOLLICULAR PHASE START" in s:
        result['stim_start_variant'] = "Follicular Phase"
    elif "CD3 STIM START" in s:
        result['stim_start_variant'] = "CD3"
    elif ("MIDLUTEAL" in s or "MID-LUTEAL" in s
          or "MID LUTEAL" in s or "LH+7" in s):
        result['stim_start_variant'] = "MidLuteal"

    # ---- PROTOCOL FAMILY FALLBACK (only used if cycletype UUID is missing) ----
    # Priority order matters: UNSTIM first (definitive), then BCPA before BCP, etc.
    #if result['unstimulated']:
    #    result['protocol_family_regex'] = "Unstimulated"
    #elif "BCPA" in s and "LUPRON OVERLAP" in s:
    #    result['protocol_family_regex'] = "BCPA Lupron Overlap"
    #elif "BCPA" in s:
    #    result['protocol_family_regex'] = "BCPA"
    #elif re.search(r'\bBCP\b', s) and "LUPRON OVERLAP" in s:
    #    result['protocol_family_regex'] = "BCP Lupron Overlap"
    #elif re.search(r'\bBCP\b', s):
    #    result['protocol_family_regex'] = "BCP"
    #elif "MLEA" in s and "LOW STIM" in s:
    #    result['protocol_family_regex'] = "MLEA Low Stim"
    #elif "MLEA" in s:
    #    result['protocol_family_regex'] = "MLEA"
    #elif "LONG ESTRACE" in s:
    #    result['protocol_family_regex'] = "Long Estrace"
    #elif "GANIRELIX" in s:
    #    result['protocol_family_regex'] = "Ganirelix"
    #elif "ANTAGONIST" in s:
    #    result['protocol_family_regex'] = "Antagonist"
    #elif "AMH IVF" in s:
    #    result['protocol_family_regex'] = "AMH"
    #elif "FALLUM" in s:
    #    result['protocol_family_regex'] = "Fallum"
    ## Standalone IUI/TDI/TIC protocols (no other stim family matched)
    #elif "LETROZOLE" in s and "CLOMID" in s:
    #    result['protocol_family_regex'] = "Letrozole + Clomid"
    #elif "LETROZOLE" in s:
    #    result['protocol_family_regex'] = "Letrozole"
    #elif "CLOMID" in s:
    #    result['protocol_family_regex'] = "Clomid"
    #elif "COH" in s:
    #    result['protocol_family_regex'] = "COH"
    #elif "NATURAL" in s:
    #    result['protocol_family_regex'] = "Natural"
    #elif result['stim_start_variant']:
    #    result['protocol_family_regex'] = f"{result['stim_start_variant']} Stim Start"

    # ---- COMPONENT FLAGS ----
    # Break the cycle_name into orthogonal fields so downstream analysis
    # can filter/group on each component independently.

    # --- Cycle purpose ---
    result['is_ivf']          = "IVF" in s
    result['is_fet']          = ("FET" in s or "FROZEN EMBRYO TRANSFER" in s
                                 or "FRESH ET" in s)
    result['is_iui']          = "IUI" in s
    result['is_tic']          = re.search(r'\bTIC\b', s) is not None
    result['is_tdi']          = re.search(r'\bTDI\b', s) is not None
    result['is_oocyte_freeze']= ("OOCYTE FREEZE" in s or "EGG FREEZE" in s
                                 or "OOCYTE: FREEZE" in s
                                 or s.startswith("OV:") or "OV MLEA" in s)
    result['is_oocyte_thaw']  = "OOCYTE THAW" in s
    result['is_embryo_thaw']  = ("EMBRYO THAW" in s or "EMBRYO REFREEZE" in s
                                 or "EMBRYO RE-FREEZE" in s)
    result['is_embryo_freeze']= "EMBRYO FREEZE" in s and "THAW" not in s
    result['is_mock']         = "MOCK CYCLE" in s or "TEST CYCLE" in s
    result['is_spontaneous']  = "SPONTANEOUS PREGNANCY" in s

    # --- Donor / third-party arrangement ---
    result['is_third_party']  = "THIRD PARTY" in s
    result['is_intended_parent']  = (re.search(r'\bIP\b', s) is not None
                                     or "INTENDED PARENT" in s)
    result['is_recipient']    = "RECIPIENT" in s
    result['is_gc']           = ("GC FET" in s or "GESTATIONAL CARRIER" in s)
    result['is_egg_donor']    = ("EGG DONOR" in s or "EGG DONATION" in s
                                 or "DONOR OOCYTE" in s)
    result['is_donor_embryo'] = "DONOR EMBRYO" in s
    result['is_reciprocal']   = "RECIPROCAL" in s
    result['is_compassionate']= "COMPASSIONATE" in s
    result['is_autologous']   = ("AUTOLOGOUS" in s
                                 and not result['is_third_party']
                                 and not result['is_egg_donor']
                                 and not result['is_donor_embryo'])

    # --- Clinic ---
    result['is_kaiser']       = "KAISER" in s

    # --- Base stim protocol (mutually exclusive — one wins) ---
    result['protocol_base'] = None
    if result['unstimulated']:
        result['protocol_base'] = "Unstimulated"
    elif "BCPA" in s:
        result['protocol_base'] = "BCPA"
    elif re.search(r'\bBCP\b', s):
        result['protocol_base'] = "BCP"
    elif "MLEA" in s:
        result['protocol_base'] = "MLEA"
    elif "LONG ESTRACE" in s:
        result['protocol_base'] = "Long Estrace"
    elif "GANIRELIX" in s:
        result['protocol_base'] = "Ganirelix"
    elif "ANTAGONIST" in s:
        result['protocol_base'] = "Antagonist"
    elif "AMH IVF" in s:
        result['protocol_base'] = "AMH"
    elif "FALLUM" in s:
        result['protocol_base'] = "Fallum"
    elif "CUSTOM SCHEDULE" in s:
        result['protocol_base'] = "Custom Schedule"

    # --- Medication / structural modifiers (each independent yes/no) ---
    result['has_lupron_overlap'] = "LUPRON OVERLAP" in s
    result['has_gonadotropins']  = "GONADOTROPINS" in s
    result['has_clomid']         = "CLOMID" in s
    result['has_letrozole']      = "LETROZOLE" in s
    result['has_menopur']        = "MENOPUR" in s
    result['has_low_stim']       = "LOW STIM" in s
    # Adjuvants: three-way — Yes / +/- / No
    if "NO ADJUVANTS" in s:
        result['adjuvants_flag'] = "No"
    elif "ADJUVANTS" in s:
        result['adjuvants_flag'] = "+/-"
    else:
        result['adjuvants_flag'] = None

    # --- FET medication route (only meaningful for FET cycles) ---
    if result['is_fet']:
        result['fet_medication'] = None
        if "TRANSDERMAL PATCH" in s or "PATCHES" in s or re.search(r'\bPATCH\b', s):
            result['fet_medication'] = "Transdermal Patches"
        elif "VAGINAL ESTRACE" in s:
            result['fet_medication'] = "Vaginal Estrace"
        elif "ORAL ESTRACE" in s or "ESTRACE ORAL" in s or "ESTRACE 2MG" in s:
            result['fet_medication'] = "Oral Estrace"
        elif "ESTRADIOL VALERATE" in s:
            result['fet_medication'] = "Estradiol Valerate"
        elif "COH" in s:
            result['fet_medication'] = "COH"
        elif ("MODIFIED NATURAL" in s or "MOD NATURAL" in s):
            result['fet_medication'] = "Modified Natural"
        elif "LETROZOLE" in s:
            result['fet_medication'] = "Letrozole"
        elif "NATURAL" in s:
            result['fet_medication'] = "Natural"
    else:
        result['fet_medication'] = None

    # --- Legacy field: build the descriptive string from the flags ---
    # Kept for backwards compatibility. Downstream code should prefer the
    # individual flag fields above.
    if result['protocol_base']:
        parts = [result['protocol_base']]
        if result['has_lupron_overlap']:      parts.append("Lupron Overlap")
        if result['has_low_stim']:            parts.append("Low Stim")
        if result['has_gonadotropins']:       parts.append("Gonadotropins")
        if result['has_clomid']:              parts.append("Clomid")
        if result['adjuvants_flag'] == "+/-": parts.append("+/- Adjuvants")
        elif result['adjuvants_flag'] == "No":parts.append("No Adjuvants")
        if result['has_menopur']:             parts.append("Menopur")
        if result['stim_start_variant']:      parts.append(f"{result['stim_start_variant']} Stim Start")
        result['protocol_family_regex'] = parts[0] if len(parts) == 1 else parts[0] + " " + " + ".join(parts[1:])
    elif result['stim_start_variant']:
        result['protocol_family_regex'] = f"{result['stim_start_variant']} Stim Start"
    elif result['is_iui'] or result['is_tic'] or result['is_tdi']:
        if result['has_letrozole'] and result['has_clomid']:
            result['protocol_family_regex'] = "Letrozole + Clomid"
        elif result['has_letrozole']:
            result['protocol_family_regex'] = "Letrozole"
        elif result['has_clomid']:
            result['protocol_family_regex'] = "Clomid"
        elif "COH" in s:
            result['protocol_family_regex'] = "COH"
        elif "NATURAL" in s:
            result['protocol_family_regex'] = "Natural"
    else:
        result['protocol_family_regex'] = None

    return result


# =============================================================================
# COMBINED PARSER
# =============================================================================

#def parse_cycle_row(row):
#    """
#    Combine UUID lookups (authoritative for cycle_type/clinic/donor/protocol_family)
#    with regex parsing (authoritative for conversions/adjuvants/freeze_all).
#
#    Also flags cases where UUID and regex disagree on protocol_family.
#    """
#    pt = PLAN_TREATMENT_MAP.get(row.get('plan_treatment'), {}) or {}
#    ct = CYCLETYPE_MAP.get(row.get('cycletype'), {}) or {}
#    rx = parse_cycle_name(row.get('cycle_name'))
#
#    # Protocol family: prefer UUID, fall back to regex
#    protocol_family = ct.get('protocol_family') or rx.get('protocol_family_regex')
#
#    # Agreement check (only when both are non-null)
#    uuid_pf = (ct.get('protocol_family') or '').lower()
#    regex_pf = (rx.get('protocol_family_regex') or '').lower()
#    if uuid_pf and regex_pf:
#        # Consider agreement if either fully contains the other's main token
#        agreement = (uuid_pf in regex_pf) or (regex_pf in uuid_pf)
#    else:
#        agreement = None
#
#    # Build a human-readable protocol_detail
#    protocol_detail_parts = []
#    if protocol_family:
#        protocol_detail_parts.append(protocol_family)
#    if rx['adjuvants']:
#        protocol_detail_parts.append(" +/- ".join(rx['adjuvants']))
#    protocol_detail = " ".join(protocol_detail_parts) if protocol_detail_parts else None
#
#    # Append conversion to protocol_detail (not cycle_type)
#    if rx['conversion_targets']:
#        conv_str = " / ".join(rx['conversion_targets'])
#        protocol_detail = (
#            f"{protocol_detail} → {conv_str}" if protocol_detail else f"→ {conv_str}"
#        )
#
#    # Classification status for QC
#
#    # Cycle types where no protocol is expected
#    NO_PROTOCOL_TYPES = {
#            'Oocyte Thaw', 'Embryo Thaw', 'Oocyte Thaw → Fresh ET',
#            'Embryo Freeze', 'Spontaneous Pregnancy', 'Mock Cycle',
#    }
#
#    if pt.get('cycle_type') is None and ct.get('protocol_family') is None:
#        status = "REVIEW: unknown UUIDs"
#    elif protocol_family is None:
#        if pt.get('cycle_type') in NO_PROTOCOL_TYPES:
#            status = "OK (no protocol expected)"
#        elif rx.get('unstimulated'):
#            status = "OK (unstimulated)"
#        else:
#            status = "REVIEW: no protocol_family"
#    elif agreement is False:
#        status = "REVIEW: UUID/regex mismatch"
#    else:
#        status = "OK"
#
#    return {
#        'cycle_type':                pt.get('cycle_type'),
#        'clinic':                    pt.get('clinic'),
#        'donor_type':                pt.get('donor_type'),
#        'protocol_family':           protocol_family,
#        'protocol_detail':           protocol_detail,
#        'adjuvants':                 rx['adjuvants'],
#        'freeze_all':                rx['freeze_all'],
#        'stim_start_variant':        rx['stim_start_variant'],
#        'conversion':                rx['conversion'],
#        'conversion_targets':        rx['conversion_targets'],
#        'original_cycle':            rx['original_cycle'],
#        'protocol_family_uuid':      ct.get('protocol_family'),
#        'protocol_family_regex':     rx['protocol_family_regex'],
#        'uuid_regex_agreement':      agreement,
#        'classification_status':     status,
#    }

def parse_cycle_row(row):
    """
    Combine UUID lookups (authoritative for cycle_type/clinic/donor/protocol_family)
    with regex parsing (authoritative for conversions/adjuvants/freeze_all).

    Also flags cases where UUID and regex disagree on protocol_family.
    """
    pt = PLAN_TREATMENT_MAP.get(row.get('plan_treatment'), {}) or {}
    ct = CYCLETYPE_MAP.get(row.get('cycletype'), {}) or {}
    rx = parse_cycle_name(row.get('cycle_name'))

    # Protocol family: prefer UUID, fall back to regex
    protocol_family = ct.get('protocol_family') or rx.get('protocol_family_regex')

    # Agreement check (only when both are non-null)
    uuid_pf = (ct.get('protocol_family') or '').lower()
    regex_pf = (rx.get('protocol_family_regex') or '').lower()
    if uuid_pf and regex_pf:
        # Consider agreement if either fully contains the other's main token
        agreement = (uuid_pf in regex_pf) or (regex_pf in uuid_pf)
    else:
        agreement = None

    # Build a human-readable protocol_detail
    protocol_detail_parts = []
    if protocol_family:
        protocol_detail_parts.append(protocol_family)
    if rx['adjuvants']:
        protocol_detail_parts.append(" +/- ".join(rx['adjuvants']))
    protocol_detail = " ".join(protocol_detail_parts) if protocol_detail_parts else None

    # Append conversion to protocol_detail (not cycle_type)
    if rx['conversion_targets']:
        conv_str = " / ".join(rx['conversion_targets'])
        protocol_detail = (
            f"{protocol_detail} → {conv_str}" if protocol_detail else f"→ {conv_str}"
        )

    # Classification status for QC

    # Cycle types where no protocol is expected
    NO_PROTOCOL_TYPES = {
            'Oocyte Thaw', 'Embryo Thaw', 'Oocyte Thaw → Fresh ET',
            'Embryo Freeze', 'Spontaneous Pregnancy', 'Mock Cycle',
    }

    if pt.get('cycle_type') is None and ct.get('protocol_family') is None:
        status = "REVIEW: unknown UUIDs"
    elif protocol_family is None:
        if pt.get('cycle_type') in NO_PROTOCOL_TYPES:
            status = "OK (no protocol expected)"
        elif rx.get('unstimulated'):
            status = "OK (unstimulated)"
        else:
            status = "REVIEW: no protocol_family"
    elif agreement is False:
        status = "REVIEW: UUID/regex mismatch"
    else:
        status = "OK"

    # Fields from UUID lookups + computed status
    result = {
        'cycle_type':                pt.get('cycle_type'),
        'clinic':                    pt.get('clinic'),
        'donor_type':                pt.get('donor_type'),
        'protocol_family':           protocol_family,
        'protocol_detail':           protocol_detail,
        'protocol_family_uuid':      ct.get('protocol_family'),
        'uuid_regex_agreement':      agreement,
        'classification_status':     status,
    }

    # Spread in ALL fields from the regex parser (medication flags,
    # cycle purpose flags, donor flags, protocol_base, etc.)
    result.update(rx)

    return result

def derive_protocol_family_broad(row):
    """
    Derive a broad cycle-type category from parsed flags and UUID lookups.
    Uses cycle_name-derived flags first; falls back to plan_treatment UUID.

    Returns one of:
        'IVF', 'IVF + FET', 'IVF + Fresh ET', 'FET', 'Fresh ET',
        'IUI', 'TIC', 'TDI',
        'Oocyte Freeze', 'Oocyte Thaw', 'Oocyte Thaw + Fresh ET',
        'Embryo Freeze', 'Embryo Thaw',
        'Mock Cycle', 'Spontaneous Pregnancy', None
    """

    # ---- Primary source: cycle_name-derived flags ----
    # Handle compound cycles first (most specific wins)
    if row.get('is_ivf') and row.get('is_fet'):
        return 'IVF + FET'
    if row.get('is_ivf') and 'FRESH ET' in str(row.get('cycle_name', '')).upper():
        return 'IVF + Fresh ET'
    if row.get('is_oocyte_thaw') and 'FRESH ET' in str(row.get('cycle_name', '')).upper():
        return 'Oocyte Thaw + Fresh ET'

    # Single-type cycles (ordered by specificity)
    if row.get('is_spontaneous'):    return 'Spontaneous Pregnancy'
    if row.get('is_mock'):           return 'Mock Cycle'
    if row.get('is_embryo_thaw'):    return 'Embryo Thaw'
    if row.get('is_embryo_freeze'):  return 'Embryo Freeze'
    if row.get('is_oocyte_thaw'):    return 'Oocyte Thaw'
    if row.get('is_oocyte_freeze'):  return 'Oocyte Freeze'
    if row.get('is_ivf'):            return 'IVF'
    if row.get('is_fet'):            return 'FET'
    if 'FRESH ET' in str(row.get('cycle_name', '')).upper(): return 'Fresh ET'
    if row.get('is_tic'):            return 'TIC'
    if row.get('is_tdi'):            return 'TDI'
    if row.get('is_iui'):            return 'IUI'

    # ---- Fallback: plan_treatment UUID lookup ----
    # Same map, different starting point when cycle_name has no info
    pt = PLAN_TREATMENT_MAP.get(row.get('plan_treatment'), {})
    ct_broad = pt.get('cycle_type')

    # Normalize plan_treatment cycle_type to match our broad categories
    if ct_broad:
        if 'IVF Freeze-All' in ct_broad or 'IVF (named' in ct_broad:
            return 'IVF'
        if ct_broad == 'IVF Fresh ET':
            return 'IVF + Fresh ET'
        if ct_broad == 'Oocyte Thaw → Fresh ET':
            return 'Oocyte Thaw + Fresh ET'
        return ct_broad   # 'FET', 'IUI', 'TIC', 'Oocyte Freeze', etc.

    return None